# 02 - Factor Research

Compute the **Alpha158** feature set (qlib, from scratch) plus classical factors, then evaluate predictive power with an **alphalens-style** IC / ICIR / turnover / quantile-returns report and a factor-decay analysis.

> **Data input:** Set exactly one of `QUANTCORTEX_PRICES_CSV` or `QUANTCORTEX_LIVE_YFINANCE=1`. Local runs also set `QUANTCORTEX_OHLCV_CSV` for Alpha158. No market data or executed outputs are committed.


In [ ]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the repository root on the path when running from research/.
for candidate in ("..", "."):
    absolute = os.path.abspath(candidate)
    if absolute not in sys.path:
        sys.path.insert(0, absolute)

from quantcortex.data.local_csv import load_ohlcv_csv, load_price_matrix

PRICE_CSV = os.environ.get("QUANTCORTEX_PRICES_CSV")
OHLCV_CSV = os.environ.get("QUANTCORTEX_OHLCV_CSV")
LIVE_YFINANCE = os.environ.get("QUANTCORTEX_LIVE_YFINANCE") == "1"

if (PRICE_CSV is not None) == LIVE_YFINANCE:
    raise RuntimeError(
        "Set exactly one of QUANTCORTEX_PRICES_CSV=/path/to/prices.csv "
        "or QUANTCORTEX_LIVE_YFINANCE=1."
    )


def load_prices(symbols, start="2018-01-01"):
    """Load adjusted closes from the explicitly selected real-data source."""
    if PRICE_CSV is not None:
        return load_price_matrix(PRICE_CSV, symbols=list(symbols), start=start)

    print(
        "Using live Yahoo Finance data through yfinance; review the provider "
        "terms and confirm that your use is permitted."
    )
    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    prices = YFinanceProvider().get_prices(list(symbols), start=start)
    if prices is None or prices.empty:
        raise RuntimeError("yfinance returned no prices")
    prices = prices.dropna(how="all").ffill(limit=5).dropna()
    if prices.shape[0] <= 120:
        raise RuntimeError("insufficient complete price history from yfinance")
    return prices


def load_ohlcv(symbol, start="2018-01-01"):
    """Load one symbol's real OHLCV data for feature engineering."""
    if OHLCV_CSV is not None:
        return load_ohlcv_csv(OHLCV_CSV, start=start)
    if not LIVE_YFINANCE:
        raise RuntimeError(
            "Set QUANTCORTEX_OHLCV_CSV=/path/to/ohlcv.csv for the Alpha158 cell."
        )

    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    frame = YFinanceProvider().fetch_ohlcv([symbol], start=start).get(symbol)
    if frame is None or frame.empty:
        raise RuntimeError(f"yfinance returned no OHLCV data for {symbol}")
    return frame.dropna()


selected_source = (
    f"local CSV {Path(PRICE_CSV).expanduser().resolve()}"
    if PRICE_CSV is not None
    else "explicit live yfinance download"
)
print(f"quantcortex research environment ready; source: {selected_source}")


In [ ]:
symbols = ["AAPL", "MSFT", "AMZN", "NVDA", "JPM", "XOM", "PG", "KO"]
prices = load_prices(symbols)
returns = prices.pct_change(fill_method=None).dropna()
print("universe:", list(prices.columns), prices.shape)

## Alpha158 features (single symbol)

In [ ]:
from quantcortex.alpha.feature_engineering.alpha158 import Alpha158

ohlcv = load_ohlcv(symbols[0])
feats = Alpha158().compute(ohlcv)
print("Alpha158 produced", feats.shape[1], "features")
assert feats.shape[1] == 158
feats.dropna().iloc[-1].head(12)


## Classical cross-sectional factors

In [ ]:
from quantcortex.alpha.factors.classical.low_vol import LowVolFactor
from quantcortex.alpha.factors.classical.momentum import MomentumFactor

mom = MomentumFactor(lookback=126, gap=21)
mom_panel = mom.compute(prices)
mom_z = mom.cross_sectional_zscore(mom_panel)
lowvol_panel = LowVolFactor(window=63).compute(prices)
print("momentum panel:", mom_panel.shape, "| non-null rows:", int(mom_panel.notna().any(axis=1).sum()))
mom_z.dropna(how="all").tail(3)

## Alphalens report (IC, ICIR, turnover, quantiles)

In [ ]:
from quantcortex.alpha.validation.alphalens_report import AlphalensReport

fwd_returns = prices.pct_change(5).shift(-5)   # 5-day forward return
factor = mom_z.dropna(how="all")
report = AlphalensReport(factor, fwd_returns, quantiles=4).compute()
turn = float(np.nanmean(np.asarray(report["turnover"], dtype=float)))
ls = float(np.asarray(report["long_short_return"]).mean())
print("IC mean  : %+.4f" % report["ic_mean"])
print("ICIR     : %+.4f" % report["icir"])
print("IC t-stat: %+.3f" % report["ic_tstat"])
print("turnover : %.3f" % turn)
print("long-short: %+.5f" % ls)

## Factor decay

In [ ]:
from quantcortex.alpha.validation.factor_decay import FactorDecay

decay = FactorDecay().compute(factor, returns, max_lag=10)
print(decay.round(4).to_string())

In [ ]:
ic = report["ic"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ic.cumsum().plot(ax=ax[0]); ax[0].set_title("Cumulative IC"); ax[0].axhline(0, color="k", lw=0.5)
decay["ic_mean"].plot(kind="bar", ax=ax[1]); ax[1].set_title("Mean IC by forward lag")
plt.tight_layout(); print("factor research complete.")